# Kathmandu traffic vision - plate label set

Cuts plates out of real Nepali vehicle photos for hand labelling; the model's readings are saved separately and not shown to the labeller.

In [ ]:
!pip install -q ultralytics

In [ ]:
"""Read a number plate: plate crop -> character boxes (chars_best.pt) -> one label per box
(glyphs_best.pt) -> text in reading order. Models come from the train-plate-reader notebook.

    python reader.py plate.jpg [--chars chars_best.pt] [--glyphs glyphs_best.pt]
"""
from __future__ import annotations

import argparse
import re

import cv2

Box = tuple  # (x1, y1, x2, y2, label, conf)


def reading_order(boxes: list[Box]) -> list[Box]:
    """Return the character boxes in the order a person reads the plate.

    Nepali plates come in one row (most car plates) or two rows (most motorcycle plates, and the
    upper line of zone + lot over the number on older red plates). Read the top row left to right,
    then the bottom row left to right.
    """
    # New row where the gap between consecutive vertical centres exceeds half a character height:
    # scales with plate size, and a tilted row's small steady steps never reach it.
    heights = sorted(b[3] - b[1] for b in boxes)
    gap = heights[len(heights) // 2] / 2
    rows, last = [], None
    for b in sorted(boxes, key=lambda b: (b[1] + b[3]) / 2):
        c = (b[1] + b[3]) / 2
        if last is None or c - last > gap:
            rows.append([])
        rows[-1].append(b)
        last = c
    return [b for row in rows for b in sorted(row, key=lambda b: b[0])]


DEV_DIGITS = set("०१२३४५६७८९")
EMB_DIGITS = set("0123456789")
DEV_PLATE = re.compile(r"L+D+L+D{1,4}")   # zone (or province), lot, vehicle class, number: बा २ च १२३४


def check_plate(labels: list[str], conf: float, min_conf: float = 0.6) -> tuple[bool, str]:
    """Is this reading plausible enough to use without a person checking it? (ok, reason)

    Catches structure errors: missed or extra characters, a letter in a number slot, mixed scripts,
    and unsure characters. It cannot catch a look-alike swapped for another of the same kind
    (३ for ६, ना for बा): those only show up as low confidence, if at all.
    """
    if not labels:
        return False, "no characters read"
    if conf < min_conf:
        return False, f"unsure character (confidence {conf:.2f} < {min_conf})"
    dev = [lab for lab in labels if not (lab in EMB_DIGITS or lab.isascii())]
    if dev and len(dev) < len(labels):
        return False, "mixes Devanagari and embossed characters"
    if dev:
        shape = "".join("D" if lab in DEV_DIGITS else "L" for lab in labels)
        if not DEV_PLATE.fullmatch(shape):
            return False, f"not a Nepali plate layout (letters/digits: {shape})"
        return True, "ok"
    # shortcut: embossed plates are only checked for digits and letters both being present;
    # add their exact layout once it is confirmed against real plates
    if not any(lab in EMB_DIGITS for lab in labels) or all(lab in EMB_DIGITS for lab in labels):
        return False, "embossed reading without both letters and digits"
    return True, "ok"


def read_plate(plate, chars, glyphs, min_conf: float = 0.25) -> tuple[list[str], float]:
    """Character labels in reading order, and the lowest confidence (a weak link makes the whole reading weak)."""
    boxes = []
    for x1, y1, x2, y2 in chars.predict(plate, imgsz=416, conf=min_conf, verbose=False)[0].boxes.xyxy.tolist():
        crop = plate[int(y1):int(y2), int(x1):int(x2)]
        if crop.size == 0:
            continue
        r = glyphs.predict(crop, imgsz=64, verbose=False)[0]
        label = r.names[r.probs.top1]
        if label != "flag":
            boxes.append((x1, y1, x2, y2, label.removeprefix("emb_"), float(r.probs.top1conf)))
    if not boxes:
        return [], 0.0
    ordered = reading_order(boxes)
    return [b[4] for b in ordered], min(b[5] for b in ordered)



In [ ]:

import csv, glob, zipfile
from pathlib import Path
import cv2
from ultralytics import YOLO
find = lambda name: sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
plates = YOLO(find("plates_best.pt")[0])
chars = YOLO(find("chars_best.pt")[0])
glyphs = YOLO([p for p in find("glyphs_best.pt") if "run-2" in p][0])     # run 2 classifier
photos = sorted(p for p in Path("/kaggle/input").rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
                and "nepali-vehicles-number-plate-dataset" in str(p))
print(len(photos), "photos")
out = Path("/kaggle/working/photos"); out.mkdir(exist_ok=True)
rows, reads = [], []
for k, photo in enumerate(photos):
    img = cv2.imread(str(photo))
    if img is None:
        continue
    pid = f"P{k + 1:04d}"
    h, w = img.shape[:2]
    s = 1100 / max(h, w)                                  # labelling copy: enough to read the plate
    cv2.imwrite(str(out / f"{pid}.jpg"), cv2.resize(img, (int(w * s), int(h * s))) if s < 1 else img,
                [cv2.IMWRITE_JPEG_QUALITY, 85])
    rows.append({"id": pid, "photo": photo.name})
    boxes = plates.predict(img, imgsz=1280, conf=0.25, verbose=False)[0].boxes.data[:, :5].tolist()
    if not boxes:
        reads.append({"id": pid, "plates_found": 0, "reading": "", "min_conf": 0, "check": "no plate found"})
        continue
    # the model's answer for the photo: the most confident plate box, with a margin
    x1, y1, x2, y2, c = max(boxes, key=lambda b: b[4])
    px, py = (x2 - x1) * 0.08, (y2 - y1) * 0.15
    crop = img[int(max(0, y1 - py)):int(min(h, y2 + py)), int(max(0, x1 - px)):int(min(w, x2 + px))]
    labels, cf = read_plate(crop, chars, glyphs)
    ok, why = check_plate(labels, cf)
    reads.append({"id": pid, "plates_found": len(boxes), "reading": "".join(labels), "min_conf": round(cf, 3),
                  "check": "ok" if ok else why})
for name, data in [("photos.csv", rows), ("model_readings.csv", reads)]:
    with open(f"/kaggle/working/{name}", "w", newline="", encoding="utf-8") as fh:
        wr = csv.DictWriter(fh, fieldnames=list(data[0]))
        wr.writeheader()
        wr.writerows(data)
with zipfile.ZipFile("/kaggle/working/photos.zip", "w") as z:
    for f in sorted(out.glob("*.jpg")):
        z.write(f, f.name)
import shutil; shutil.rmtree(out)
print(len(rows), "photos;", sum(r["plates_found"] > 1 for r in reads), "with more than one plate box;",
      sum(r["plates_found"] == 0 for r in reads), "with none;", sum(r["check"] == "ok" for r in reads), "readings pass the format check")
